In [ ]:
from datetime import datetime
from datetime import timedelta

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import os
import json
import pickle
from tqdm.notebook import tqdm
import plotly.express as px
import plotly


import sklearn
from sklearn.manifold import TSNE, Isomap

from sklearn.preprocessing import StandardScaler 
from sklearn.preprocessing import LabelEncoder
from sklearn.mixture import GaussianMixture

from sklearn.model_selection import GridSearchCV
from sklearn.cluster import (KMeans, AgglomerativeClustering, DBSCAN)

from sklearn.metrics import silhouette_score

from utils.portfolio import MarkowitzPortfolio
from utils.portfolio_metrics import (calculate_measures, show_drawdown_recovery, 
                                     find_max_recovery, find_max_drawdown)

from utils.pipeline import (general_pipeline, calc_metrics, clustering_estimation, 
                            backtesting_one_model, select_assets, clustering_estimation, get_clusters)

import warnings
warnings.filterwarnings("ignore")

In [ ]:
with open('config/config.json', 'r') as file:
    config = json.load(file)
    
with open('config/config_emb_path.json', 'r') as file:
    config_emb = json.load(file)
    
rs = config['random_state']

year_start = 2021
fine_tune = True

for key in config_emb.keys():
    config_emb[key] = config_emb[key].format(year_start)

# data loading

In [ ]:
df_original = pd.read_csv(config['ticker_data_close'], index_col=0)

df_na = df_original.isna().sum()
thresh = 0.05 * len(df_original)
stocks_to_drop = df_na[df_na > thresh].index.tolist()
df_original = df_original.drop(stocks_to_drop, axis=1)
df_original = df_original.dropna(axis=0)

df_original.index = pd.to_datetime(df_original.index)
df_original_train = df_original[(df_original.index < datetime(year_start, 1, 1)) ]
df_original_train.tail()

In [ ]:
df = pd.read_csv(config['ticker_data_preprocessed'], index_col=0)
print(df.shape)
df.head()

In [ ]:
df_pct = df.drop(['sector'], axis=1).T
df_pct.index = pd.to_datetime(df_pct.index)

tickers_list = df_pct.columns.tolist()

df_pct.head()

In [ ]:
df_pct_train = df_pct[(df_pct.index < datetime(year_start, 1, 1)) ]

In [ ]:
df_market = pd.read_csv(config['ticker_data_sp500'], index_col=0)
df_market.index = pd.to_datetime(df_market.index)
df_market = df_market.pct_change()
df_market.plot()

In [ ]:
df_sectors = pd.read_csv(config['tickers_sectors_path'], index_col=0)
df_sectors = df_sectors.sort_values('ticker')
df_sectors.head()

In [ ]:
tickers = df.index.tolist()

In [ ]:
# feats_tsne = TSNE(n_components=2).fit_transform(df.drop(['sector'], axis=1))
# df_tsne = pd.DataFrame({'axis0':feats_tsne[:, 0],'axis1':feats_tsne[:, 1],'sector':df['sector']})

# fig = px.scatter(df_tsne, x = 'axis0', y = 'axis1', color="sector", width=800, height=600)
# fig.show()

# Experiments

In [ ]:
def custom_score_clustering_gridsearch(estimator, X, y=None):
    estimator.fit(X)
    labels_predicted = estimator.labels_
    score = silhouette_score(X, labels_predicted)
    return score

def selection_sharp(list_tickers, n_save, df_pct, riskfree_rate):
    df_pct = df_pct[list_tickers]
    
    sharp = (df_pct.mean() - riskfree_rate)/df_pct.std()
    selected_tickers = sharp.sort_values(ascending=False).head(n_save).index.tolist()
    
    return selected_tickers

# embeddings

In [ ]:
emb_dict = dict()

# for name, path in config_emb.items():
#     emb_dict[name] = pd.read_csv(path, index_col=0).values

top_path = f"data/embeddings_{year_start}"
all_files = os.listdir(top_path)

for emb_path in all_files:
    
    emb_name = emb_path.replace('.csv', '')
    emb_path = top_path + '/' + emb_path
    
    emb_dict[emb_name] = pd.read_csv(emb_path, index_col=0).values

In [ ]:
for method_name, data in emb_dict.items():
    print(method_name, np.sum(np.isnan(data)), data.shape)

# pipeline

In [ ]:
clust_models_init = {'KMeans': KMeans(n_clusters=config['n_clusters'], random_state=rs),
                     'Agg': AgglomerativeClustering(n_clusters=config['n_clusters']),
                     #'DBSCAN': DBSCAN(),
                    }

clust_models_params = {'KMeans': {
                   'n_clusters':np.arange(9, 14),
                   'init': ['k-means++', 'random'],
                   'algorithm':['auto', 'full', 'elkan']
                },
                     'Agg': {
                   'n_clusters':np.arange(9, 14),
                   'linkage': ['ward', 'complete', 'average', 'single'],
                },
                     'DBSCAN':{
                   'n_clusters':np.arange(9, 14),
                   'init': ['k-means++', 'random'],
                   'algorithm':['auto', 'full', 'elkan']
                }
                }

In [ ]:
riskfree = config['riskless_rate']
riskfree_rate=(1 + riskfree) **(1/252) - 1,
ret_det=(1 + 0.03) **(1/252) - 1,
    
    
clust_params = {#'clust_model':clust_models_init[clust_model_name],
                'make_grid':fine_tune, 
                #'grid_params':clust_models_params[clust_model_name],
                'grid_metric':custom_score_clustering_gridsearch}


selection_params = {'selection_method':selection_sharp,
                    'n_save':2, 
                    'riskfree_rate':riskfree_rate,}

backtesting_params = {'port_model':MarkowitzPortfolio,
                      'window_train':24, # size of train window in months
                      'window_test':1,  # size of train window in months
                       
                      'test_start_year':year_start, #start data year
                      'test_start_month':1, #start data month 
                      'test_finish_year':2022, #end data year
                      'test_finish_month':11, #end data month
                      'ret_det':ret_det
                     }

In [ ]:
dict_port_methods = dict() #??????? ???????? ????????? ??? ???? ??????? ?????????????
dict_weight_methods = dict()
cluster_metrics_df = pd.DataFrame() 
financial_metrics_df = pd.DataFrame() 


for emb_model_name, embedding_data in tqdm(emb_dict.items()):
    ports_df = pd.DataFrame()
    dict_weights = dict()
    
    
    for clust_model_name, clust_model in clust_models_init.items():
        
        clust_params['clust_model'] = clust_model
        clust_params['grid_params'] = clust_models_params[clust_model_name]
    
        weights_all, return_portfolio, cluster_metrics, fin_metrics = general_pipeline(
            df_pct,
            df_market,
            df_sectors, 
            embedding_data=embedding_data,
            clust_params=clust_params,
            selection_params=selection_params,
            backtesting_params=backtesting_params)
        
        cluster_metrics['clust_model'] = clust_model_name
        fin_metrics['clust_model'] = clust_model_name
        
        cluster_metrics['emb_model'] = emb_model_name
        fin_metrics['emb_model'] = emb_model_name
        
        cluster_metrics_df = pd.concat([cluster_metrics_df, pd.DataFrame(cluster_metrics).T], axis=0)
        financial_metrics_df = pd.concat([financial_metrics_df, pd.DataFrame(fin_metrics).T], axis=0)  
        
        ports_df[clust_model_name] = return_portfolio
        dict_weights[clust_model_name] = weights_all
        
    dict_port_methods[emb_model_name] = ports_df
    dict_weight_methods[emb_model_name] = dict_weights

In [ ]:
financial_metrics_df.to_csv(config['financial_metric_path'].format(year_start, fine_tune))
cluster_metrics_df.to_csv(config['clust_metric_path'].format(year_start, fine_tune))

with open(config['port_path'].format(year_start, fine_tune), 'wb') as f:
    pickle.dump(dict_port_methods, f)
    
with open(config['weights_path'].format(year_start, fine_tune), 'wb') as f:
    pickle.dump(dict_weight_methods, f)

### Baseline clustering

In [ ]:
from tslearn.clustering import TimeSeriesKMeans, KShape
from utils.MSTcorr import MSTcorrClustering

In [ ]:
tslearn_models = {'MSTcorr': MSTcorrClustering(n_clusters=config['n_clusters']),
                  'KShape' : KShape(n_clusters=config['n_clusters'], n_init=1, random_state=rs),
                  'TimeSeriesKMeans' : TimeSeriesKMeans(n_clusters=config['n_clusters'], metric="dtw", 
                                                        random_state=rs, n_jobs=4),
                  
                 }


tslearn_params = {
    'TimeSeriesKMeans' : {
                   'n_clusters':np.arange(9, 14),
                   'init': ['k-means++', 'random'],
                },
    
     'MSTcorr': {  
         'n_clusters':np.arange(9, 14),
         'linkage': ['complete', 'average', 'single'],
                },

    'KShape' : {'n_clusters':np.arange(9, 14),
                'tol': [1e-4, 1e-5, 1e-6],
                }
    
}

In [ ]:
clust_params = {'make_grid':fine_tune, 
                'grid_metric':custom_score_clustering_gridsearch}

for clust_model_name, clust_model in tqdm(tslearn_models.items()):
    
    emb_model_name = clust_model_name
    clust_params['clust_model'] = clust_model
    clust_params['grid_params'] = tslearn_params[clust_model_name]
    
    ports_df = pd.DataFrame()
    dict_weights = dict()
    
    weights_all, return_portfolio, cluster_metrics, fin_metrics = general_pipeline(
        df_pct,
        df_market,
        df_sectors, 
        embedding_data=df_original_train.T.values,
        clust_params=clust_params,
        selection_params=selection_params,
        backtesting_params=backtesting_params)
    
    
    cluster_metrics['clust_model'] = clust_model_name
    fin_metrics['clust_model'] = clust_model_name

    cluster_metrics['emb_model'] = emb_model_name
    fin_metrics['emb_model'] = emb_model_name

    cluster_metrics_df = pd.concat([cluster_metrics_df, pd.DataFrame(cluster_metrics).T], axis=0)
    financial_metrics_df = pd.concat([financial_metrics_df, pd.DataFrame(fin_metrics).T], axis=0)  

    ports_df[clust_model_name] = return_portfolio
    dict_weight_methods[clust_model_name] = weights_all

dict_port_methods[emb_model_name] = ports_df
dict_weight_methods[emb_model_name] = dict_weights

In [ ]:
financial_metrics_df.to_csv(config['financial_metric_path'].format(year_start, fine_tune))
cluster_metrics_df.to_csv(config['clust_metric_path'].format(year_start, fine_tune))

with open(config['port_path'].format(year_start, fine_tune), 'wb') as f:
    pickle.dump(dict_port_methods, f)
    
with open(config['weights_path'].format(year_start, fine_tune), 'wb') as f:
    pickle.dump(dict_weight_methods, f)

### benchmarks

In [ ]:
# index SP_500

clust_model_name = 'sp500'
emb_model_name = 'sp500'

fin_metrics= calc_metrics(df_market.loc[dict_port_methods['ts2vec'].index] .copy(), df_market, 
                                            riskfree_rate=riskfree_rate)
fin_metrics['clust_model'] = clust_model_name
fin_metrics['emb_model'] = emb_model_name

financial_metrics_df = pd.concat([financial_metrics_df, pd.DataFrame(fin_metrics).T], axis=0)  

In [ ]:
# economic sectors
clust_model_name = 'sectors'
emb_model_name = 'sectors'

clust_econ_sectors = LabelEncoder().fit_transform(df['sector'])
df_clusters = pd.DataFrame([tickers_list, clust_econ_sectors], index=['ticker', 'cluster']).T


selected_tickers = select_assets(df_clusters, df_pct, **selection_params)
df_pct_loc = df_pct[selected_tickers]
weights_all, return_portfolio = backtesting_one_model(df_pct_loc, # df with pct_changes: columns - tick, index - date
                    **backtesting_params)
cluster_metrics = clustering_estimation(df_pct.T.values, df_clusters, df_sectors)
fin_metrics = calc_metrics(return_portfolio.copy(), df_market, riskfree_rate=riskfree_rate)


cluster_metrics['clust_model'] = clust_model_name
fin_metrics['clust_model'] = clust_model_name

cluster_metrics['emb_model'] = emb_model_name
fin_metrics['emb_model'] = emb_model_name

cluster_metrics_df = pd.concat([cluster_metrics_df, pd.DataFrame(cluster_metrics).T], axis=0)
financial_metrics_df = pd.concat([financial_metrics_df, pd.DataFrame(fin_metrics).T], axis=0)  

ports_df[clust_model_name] = return_portfolio
dict_weight_methods[clust_model_name] = weights_all

dict_port_methods[emb_model_name] = ports_df
dict_weight_methods[emb_model_name] = dict_weights

# results estimation

In [ ]:
financial_metrics_df.sort_values('Sharpe', ascending=False)

In [ ]:
cluster_metrics_df.sort_values(['Sil'], ascending=False)

In [ ]:
financial_metrics_df.to_csv(config['financial_metric_path'].format(year_start, fine_tune))
cluster_metrics_df.to_csv(config['clust_metric_path'].format(year_start, fine_tune))

with open(config['port_path'].format(year_start, fine_tune), 'wb') as f:
    pickle.dump(dict_port_methods, f)
    
with open(config['weights_path'].format(year_start, fine_tune), 'wb') as f:
    pickle.dump(dict_weight_methods, f)